<a id="top"></a>
# Marketing bancario: business & data understanding

Questo file accompagna il lavoro di definizione del problema e di prima esplorazione dati per una campagna di marketing telefonico su conti deposito di una banca retail.

<a id="indice"></a>
## Indice

- [Contesto e obiettivo del progetto](#contesto-obiettivo)
- [Da dove arrivano i dati](#origine-dati)
- [Caricamento del dataset](#caricamento-dataset)
- [Primo sguardo ai dati](#primo-sguardo)
- [Esplorazioni mirate alle domande di business](#eda-business)
- [VS Code Data Wrangler: uno sguardo rapido ai dati](#data-wrangler)
- [Punti chiave da portare al modello](#sintesi)


<a id="contesto-obiettivo"></a>
## Contesto e obiettivo del progetto

La banca sta portando avanti campagne di telemarketing per proporre ai clienti un **deposito a termine**. Ogni campagna è fatta di telefonate: in alcune il cliente accetta il prodotto, nella maggior parte dei casi rifiuta.

La direzione marketing chiede di:

- capire quali profili di clienti rispondono meglio;
- stimare la probabilità di sottoscrizione **prima di chiamare**;
- usare questa informazione per selezionare a chi telefonare per primo e come allocare il budget.

Traduciamo questo bisogno in un task di **classificazione binaria**: dato un cliente e il contesto della chiamata, vogliamo prevedere se sottoscriverà (`y = "yes"`) oppure no (`y = "no"`).

In questo notebook ci concentriamo sulla parte congiunta di **business understanding** e **data understanding**: usiamo le domande del business per leggere i dati e, allo stesso tempo, usiamo i dati per affinare e rendere più precise le domande.


<a id="origine-dati"></a>
## Da dove arrivano i dati

Per questo progetto abbiamo lavorato con il team che gestisce il data warehouse della banca. Loro ci hanno esposto una **vista logica** che mette insieme informazioni provenienti da più aree:

- **Anagrafica cliente** (ufficio CRM): età, professione, stato civile, livello di istruzione.
- **Rischio e crediti** (credit risk): presenza di crediti in default, saldo medio annuo, mutui e prestiti personali attivi.
- **Log delle campagne** (marketing): tipo di contatto, giorno e mese della chiamata, durata dell’ultima telefonata, numero di contatti fatti finora, storia delle campagne passate.

Il risultato è una tabella a livello di **cliente × ultima chiamata**, in cui ogni riga rappresenta un contatto della campagna e la colonna `y` indica se il cliente ha sottoscritto il deposito a termine.

Le principali variabili disponibili sono:

| Gruppo | Variabile | Descrizione sintetica |
|--------|-----------|-----------------------|
| Cliente | `age` | Età del cliente (anni) |
| Cliente | `job` | Tipo di lavoro (management, technician, student, …) |
| Cliente | `marital` | Stato civile (single, married, divorced) |
| Cliente | `education` | Livello di istruzione (primary, secondary, tertiary, …) |
| Rischio/crediti | `default` | Credito in default (yes/no) |
| Rischio/crediti | `balance` | Saldo medio annuo del conto (euro) |
| Rischio/crediti | `housing` | Presenza di mutuo (yes/no) |
| Rischio/crediti | `loan` | Presenza di prestito personale (yes/no) |
| Campagna attuale | `contact` | Mezzo di contatto (cellular, telephone) |
| Campagna attuale | `day_of_week` | Giorno della settimana dell’ultimo contatto |
| Campagna attuale | `month` | Mese dell’ultimo contatto |
| Campagna attuale | `duration` | Durata dell’ultima chiamata (secondi) |
| Campagna attuale | `campaign` | Numero di chiamate in questa campagna per il cliente |
| Campagne passate | `pdays` | Giorni dall’ultima chiamata in campagne precedenti (-1 = mai contattato) |
| Campagne passate | `previous` | Numero di contatti in campagne precedenti |
| Campagne passate | `poutcome` | Esito della campagna precedente (success, failure, other, …) |
| Target | `y` | Il cliente ha sottoscritto il deposito a termine? (yes/no) |

Questa tabella è quello che il business sa di poter ottenere in modo sistematico su larga scala; la nostra analisi e il futuro modello dovranno rispettare questo perimetro informativo.


<a id="caricamento-dataset"></a>
## Caricamento del dataset

Per la parte di laboratorio usiamo la libreria `ucimlrepo`, che espone i dataset del repository UCI in forma di `pandas.DataFrame`. Il dataset che useremo è **Bank Marketing** (id 222 nel catalogo UCI).


In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import plotly.express as px

# Caricamento del dataset Bank Marketing (id=222 nel repository UCI)
bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features
y = bank_marketing.data.targets

# Costruiamo un unico DataFrame con feature e target
df = X.copy()
if isinstance(y, pd.DataFrame):
    if y.shape[1] == 1:
        df["y"] = y.iloc[:, 0]
    else:
        for col in y.columns:
            df[col] = y[col]
else:
    df["y"] = y

print("Dimensione feature:", X.shape)
print("Dimensione target:", y.shape)
df.head()


Dimensione feature: (45211, 16)
Dimensione target: (45211, 1)


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


<a id="primo-sguardo"></a>
## Primo sguardo ai dati

Partiamo da controlli essenziali: dimensione del dataset, tipi delle colonne, presenza di valori mancanti e distribuzione del target `y`. Questi passaggi servono sia al data scientist sia al business: chiariscono **quanto è grande il problema** e **quanto sono affidabili i dati di partenza**.


In [4]:
# Struttura del dataset
df.info()

<class 'pandas.DataFrame'>
Index: 12790 entries, 1 to 45205
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   age          12790 non-null  int64
 1   job          12722 non-null  str  
 2   marital      12790 non-null  str  
 3   education    12262 non-null  str  
 4   default      12790 non-null  str  
 5   balance      12790 non-null  int64
 6   housing      12790 non-null  str  
 7   loan         12790 non-null  str  
 8   contact      9411 non-null   str  
 9   day_of_week  12790 non-null  int64
 10  month        12790 non-null  str  
 11  duration     12790 non-null  int64
 12  campaign     12790 non-null  int64
 13  pdays        12790 non-null  int64
 14  previous     12790 non-null  int64
 15  poutcome     2579 non-null   str  
 16  y            12790 non-null  str  
dtypes: int64(7), str(10)
memory usage: 2.3 MB


In [5]:
# Valori mancanti per colonna (in percentuale)
missing_ratio = df.isna().mean().sort_values(ascending=False) * 100
print("\nPercentuale di valori mancanti per colonna:")
print(missing_ratio.round(2))


Percentuale di valori mancanti per colonna:
poutcome       79.84
contact        26.42
education       4.13
job             0.53
age             0.00
default         0.00
balance         0.00
housing         0.00
marital         0.00
loan            0.00
day_of_week     0.00
duration        0.00
month           0.00
campaign        0.00
pdays           0.00
previous        0.00
y               0.00
dtype: float64


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Nota importante**

Nel dataset originale la variabile `duration` (durata dell’ultima chiamata) è fortemente correlata con `y`: chiamate più lunghe tendono a portare più spesso a una sottoscrizione. Tuttavia `duration` è nota **solo a chiamata già avvenuta**.

Per un modello che deve decidere chi chiamare **prima** della telefonata, `duration` è una forma di *data leakage*: descrive un effetto della risposta del cliente, non una vera informazione predittiva disponibile in anticipo. Useremo `duration` nell’EDA per capire il comportamento della campagna, ma la terremo sotto osservazione quando passeremo alla fase di modellazione.

</div>


Per capire l’ordine di grandezza del problema, guardiamo quanti clienti hanno effettivamente sottoscritto il deposito a termine e quanto è sbilanciata la variabile `y`.


In [8]:
# valuta la ccorrelazione tra y e duration, ma codifica y in 0 e 1
y_numeric = df["y"].map({"no": 0, "yes": 1})
corr_duration_y = df["duration"].corr(y_numeric)
print(f"Correlazione tra duration e y (numerica): {corr_duration_y:.4f}")



Correlazione tra duration e y (numerica): 0.3979


In [7]:
# Converte y in numerica: no=0, yes=1
y_numeric = df["y"].map({"no": 0, "yes": 1})

# Correlazione di Pearson (equivalente alla correlazione point-biserial in questo caso)
corr_duration_y = df["duration"].corr(y_numeric)

print(f"Correlazione tra duration e y (numerica): {corr_duration_y:.4f}")

Correlazione tra duration e y (numerica): 0.3979


In [6]:
# Distribuzione del target y
target_counts = df["y"].value_counts().reset_index()
target_counts.columns = ["y", "count"]
target_counts["percentage"] = target_counts["count"] / target_counts["count"].sum() * 100

print(target_counts)

fig = px.bar(
    target_counts,
    x="y",
    y="count",
    text=target_counts["percentage"].map(lambda v: f"{v:.1f}%"),
    title="Distribuzione della risposta alla campagna (y)"
)
fig.update_layout(
    xaxis_title="Ha sottoscritto il deposito a termine?",
    yaxis_title="Numero di clienti"
)
fig.show()


     y  count  percentage
0   no  10878   85.050821
1  yes   1912   14.949179


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Attenzione**

La forte asimmetria tra classi significa che un modello banale che predice sempre `"no"` può ottenere un’accuratezza apparente molto alta. Nelle fasi successive useremo metriche e procedure di valutazione più robuste rispetto alla semplice accuracy per evitare di cadere in questa trappola.

</div>


<a id="eda-business"></a>
## Esplorazioni mirate alle domande di business

In questa fase non stiamo ancora costruendo un modello. Stiamo usando i dati per rispondere a domande pratiche del marketing, ad esempio:

- quali segmenti di clienti convertono meglio?
- quanto pesa la storia delle campagne passate?
- il volume di contatti rischia di diventare insistente e poco efficace?

Le visualizzazioni che seguono sono pensate come spunti per questa conversazione, non come analisi esaustive.


### Profilo demografico: età

Un primo controllo naturale riguarda l’età: la campagna sta parlando al segmento giusto? Usiamo un istogramma, colorato per esito della campagna.


In [9]:
# Distribuzione dell'età per esito della campagna
fig = px.histogram(
    df,
    x="age",
    color="y",
    nbins=30,
    barmode="overlay",
    marginal="box",
    title="Distribuzione dell'età per esito della campagna"
)
fig.update_layout(
    xaxis_title="Età",
    yaxis_title="Numero di clienti"
)
fig.show()


Di solito si osserva una forte concentrazione di contatti nelle fasce centrali (adulti in età lavorativa). Se nel grafico compaiono molti contatti giovanissimi o molto anziani, è un buon punto da discutere con il marketing: stiamo davvero parlando alle persone con maggiore probabilità di sottoscrivere?


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

costruisci un grafico analogo con le dirtiibuzioni a confronto per:

- `marital` (stato civile), oppure
- `education` (livello di istruzione).

</div>


### Segmentazione per professione

Il marketing ha spesso in mente “profili ideali” (studenti, pensionati, lavoratori dipendenti, autonomi). Possiamo misurare quanto questi profili rispondono davvero usando il tasso di sottoscrizione per tipo di lavoro (`job`).


In [10]:
# Tasso di sottoscrizione per tipo di lavoro
job_summary = (
    df.groupby("job")["y"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

# Ci aspettiamo colonne "no" e "yes"
for col in ["no", "yes"]:
    if col not in job_summary.columns:
        job_summary[col] = 0

job_summary["total"] = job_summary["no"] + job_summary["yes"]
job_summary["conversion_rate"] = job_summary["yes"] / job_summary["total"]

job_summary_sorted = job_summary.sort_values("conversion_rate", ascending=False)
job_summary_sorted


y,job,no,yes,total,conversion_rate
8,student,620,258,878,0.293850
10,unemployed,323,78,401,0.194514
6,self-employed,360,86,446,0.192825
3,housemaid,120,24,144,0.166667
4,management,2469,478,2947,0.162199
0,admin.,1493,235,1728,0.135995
9,technician,2273,347,2620,0.132443
5,retired,95,13,108,0.120370
7,services,1057,141,1198,0.117696
2,entrepreneur,211,27,238,0.113445


In [11]:
fig = px.bar(
    job_summary_sorted,
    x="job",
    y="conversion_rate",
    hover_data=["total", "yes", "no"],
    title="Tasso di sottoscrizione per tipo di lavoro"
)
fig.update_layout(
    xaxis_title="Tipo di lavoro",
    yaxis_title="Tasso di sottoscrizione"
)
fig.update_yaxes(tickformat=".1%")
fig.show()


Qui interessa soprattutto il **tasso di sottoscrizione**, non solo il volume di contatti. Un job molto rappresentato ma con tasso basso potrebbe essere un segmento da ridurre nelle prossime campagne; l’opposto vale per job poco contattati ma con tasso alto.


### Intensità della campagna: quante volte chiamiamo lo stesso cliente?

La variabile `campaign` conta quante chiamate sono state fatte a quello stesso cliente nella campagna corrente. Una domanda tipica:

> dopo quante chiamate il ritorno si appiattisce o diventa negativo?

Per iniziare guardiamo la distribuzione di `campaign` e come varia rispetto a `y`.


In [12]:
# Distribuzione del numero di chiamate per cliente (campaign)
df["campaign_clipped"] = df["campaign"].clip(upper=10)

fig = px.histogram(
    df,
    x="campaign_clipped",
    color="y",
    barmode="group",
    title="Numero di chiamate per cliente (valori > 10 mostrati come 10+)"
)
fig.update_layout(
    xaxis_title="Numero di chiamate nella campagna (10 = 10 o più)",
    yaxis_title="Numero di clienti"
)
fig.show()


Una forte concentrazione su 1–2 chiamate è attesa; eventuali code molto lunghe su 8–10 contatti sullo stesso cliente sono un segnale da discutere: quanto costa insistere e che ritorno abbiamo?


### Storia delle campagne: imparare dal passato

Le variabili `previous`, `pdays` e `poutcome` descrivono come il cliente è stato gestito nelle campagne precedenti. In particolare:

- `previous`: quante volte è stato contattato prima;
- `poutcome`: com’è andata l’ultima campagna su di lui.

Un’ipotesi naturale è che chi ha già risposto positivamente in passato sia più propenso a rispondere di nuovo.


In [13]:
# Tasso di sottoscrizione per esito della campagna precedente
poutcome_summary = (
    df.groupby("poutcome")["y"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["no", "yes"]:
    if col not in poutcome_summary.columns:
        poutcome_summary[col] = 0

poutcome_summary["total"] = poutcome_summary["no"] + poutcome_summary["yes"]
poutcome_summary["conversion_rate"] = poutcome_summary["yes"] / poutcome_summary["total"]

poutcome_summary_sorted = poutcome_summary.sort_values("conversion_rate", ascending=False)
poutcome_summary_sorted


y,poutcome,no,yes,total,conversion_rate
2,success,185,338,523,0.646272
1,other,515,115,630,0.182540
0,failure,1209,217,1426,0.152174


Questo tipo di analisi aiuta il marketing a definire priorità: ad esempio decidere se contattare per primi i clienti con una precedente campagna di successo, oppure concentrarsi sui mai contattati ma con caratteristiche anagrafiche favorevoli.


<a id="data-wrangler"></a>
## VS Code Data Wrangler: uno sguardo rapido ai dati

Per alcune domande veloci (filtri, ordinamenti, semplici aggregazioni) può essere comodo usare uno strumento visuale.

Nel corso useremo **VS Code Data Wrangler** su questo stesso dataset per:

- filtrare i clienti per segmenti specifici (es. solo `job == "student"` con `balance > 0`);
- verificare rapidamente i valori distinti di una variabile;
- generare trasformazioni semplici e poi esportare il codice Python corrispondente.

In questa fase il Data Wrangler non sostituisce l’EDA in Python, ma la affianca come strumento per ragionare insieme al business di fronte ai dati “vivi”.


<a id="sintesi"></a>
## Punti chiave da portare al modello

Dal punto di vista di business e di dati, le evidenze preliminari da tenere a mente sono:

- il target `y` è sbilanciato: i “no” sono molti di più dei “yes`;
- alcune variabili (come `duration`) sono fortemente informative ma non utilizzabili in modo realistico per decidere **prima** di chiamare;
- esistono segmenti con tassi di risposta molto diversi (per job, per storia di campagna), utili per costruire modelli e per definire semplici regole di business;
- la storia delle campagne precedenti (`poutcome`, `previous`) sembra contenere informazione preziosa per la priorità di contatto.

Nella fase successiva useremo queste osservazioni per impostare le trasformazioni e le scelte di feature in vista del primo modello di machine learning.
